### Overview

Cross-validation of raw counts-based machine learning models using the SCAN-B HiSeq training set (80%)

Raw count normalization pipeline: PyDESeq2 median of ratios normalization --> Log2 transformation --> Standardization

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from pydeseq2.preprocessing import deseq2_norm_fit, deseq2_norm_transform
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
import random

### 1. Import and prepare input data

In [2]:
train_data = pd.read_csv("C:/Users/User/Documents/master_thesis_project_analysis/datasets/SCANB_GSE202203/scanb_hiseq_train_test_sets/train_test_80_20/SCANB_HiSeq_pam50gene_raw_counts_subtype_train_80.csv", 
                          header=0, index_col=0)

In [3]:
train_data.shape

(2204, 52)

In [4]:
X_train = train_data.iloc[:, 0:50]
Y_train = train_data.iloc[:, [50]]

# check if the indices of x and y training sets match
X_train.index.equals(Y_train.index)

True

In [5]:
# label encoding of Y_train
label_encoder = LabelEncoder()
Y_train['subtype'] = label_encoder.fit_transform(Y_train['subtype'])

# check class count before label encoding
print("Class count before label encoding")
print(train_data['subtype'].value_counts())

# check class count after label encoding
print("\nClass count after label encoding")
print(Y_train['subtype'].value_counts())

Class count before label encoding
subtype
LumA     1119
LumB      654
Basal     230
Her2      201
Name: count, dtype: int64

Class count after label encoding
subtype
2    1119
3     654
0     230
1     201
Name: count, dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_11900\4218632629.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Y_train['subtype'] = label_encoder.fit_transform(Y_train['subtype'])


### 2. Cross-validation of model performance

In [6]:
# define seed
seed = 42

# define 5-fold stratified cross-validation
random.seed(seed)
np.random.seed(seed)
stratified_k_fold = StratifiedKFold(n_splits=5, shuffle = True, random_state = seed)

#### 2.1 Random Forest

In [7]:
random.seed(seed)
np.random.seed(seed)

fold = 1
f1_pam50_rf = []
acc_pam50_rf = []
prec_pam50_rf = []
recall_pam50_rf = []
mcc_pam50_rf = []

for train_index, val_index in stratified_k_fold.split(Y_train, Y_train):
    print(f"Starting fold {fold}\n")
    
    # split the training set into train and validation sets for 5-fold stratified cross validation
    print(f"Splitting training set into train and validation sets...")
    x_train, x_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train, y_val = Y_train.iloc[train_index], Y_train.iloc[val_index]

    print(f"Shape of X_train and X_validation sets before filtering: {x_train.shape} and {x_val.shape}")
    print(f"Shape of y_train and y_validation sets before filtering: {y_train.shape} and {y_val.shape}")

     # perform pydeseq2 median of ratios normalization
    logMeans, filteredGenes = deseq2_norm_fit(x_train)
    x_train_deseq2 = deseq2_norm_transform(x_train, logMeans, filteredGenes)[0]
    x_val_deseq2 = deseq2_norm_transform(x_val, logMeans, filteredGenes)[0]

    # perform log2 transformation
    x_train_log2 = np.log2(x_train_deseq2 + 1)
    x_val_log2 = np.log2(x_val_deseq2 + 1)
   
    # perform standardization
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train_log2)
    x_val_scaled = scaler.transform(x_val_log2)

    x_train_scaled = pd.DataFrame(x_train_scaled, index=x_train_log2.index, columns=x_train_log2.columns)
    x_val_scaled = pd.DataFrame(x_val_scaled, index=x_val_log2.index, columns=x_val_log2.columns)
   
    print(f"Shape of X_train and X_validation sets: {x_train_scaled.shape} and {x_val_scaled.shape}")
    print(f"Shape of y_train and y_validation sets: {y_train.shape} and {y_val.shape}")
    print(f"Columns of x train and x val match: {x_train_scaled.columns.equals(x_val_scaled.columns)}")
     
    # build rf classifier
    rfc = RandomForestClassifier(n_estimators=200, min_samples_leaf=1, min_samples_split=4, max_depth=None, random_state=seed)

    # fit the rf classifier on the training fold sets
    rfc.fit(x_train_scaled, y_train.values.ravel())

    # make predictions on the validation fold set
    y_pred = rfc.predict(x_val_scaled)

    # calculate metric scores
    mcc = metrics.matthews_corrcoef(y_val.values.ravel(), y_pred)
    f1 = metrics.f1_score(y_val.values.ravel(), y_pred, average='macro')
    recall = metrics.recall_score(y_val.values.ravel(), y_pred, average='macro')
    precision = metrics.precision_score(y_val.values.ravel(), y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_val.values.ravel(), y_pred)
    
    print(metrics.classification_report(y_val.values.ravel(), y_pred, digits=4))

    mcc_pam50_rf.append(mcc)
    f1_pam50_rf.append(f1)
    recall_pam50_rf.append(recall)
    prec_pam50_rf.append(precision)
    acc_pam50_rf.append(accuracy)

    fold += 1


Starting fold 1

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets before filtering: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets before filtering: (1763, 1) and (441, 1)
Shape of X_train and X_validation sets: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
Columns of x train and x val match: True
              precision    recall  f1-score   support

           0     1.0000    0.9783    0.9890        46
           1     1.0000    0.8500    0.9189        40
           2     0.9643    0.9643    0.9643       224
           3     0.8986    0.9466    0.9219       131

    accuracy                         0.9501       441
   macro avg     0.9657    0.9348    0.9485       441
weighted avg     0.9517    0.9501    0.9502       441

Starting fold 2

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets before filtering: (1763, 50) and (441, 50)
Shap

In [8]:
print(f'Mean Accuracy: {round(np.mean(acc_pam50_rf), 4)} \u00B1 {round(np.std(acc_pam50_rf), 4)}')
print(f'Mean Precision: {round(np.mean(prec_pam50_rf), 4)} \u00B1  {round(np.std(prec_pam50_rf), 4)}')
print(f'Mean Recall: {round(np.mean(recall_pam50_rf), 4)} \u00B1  {round(np.std(recall_pam50_rf), 4)}')
print(f'Mean F1: {round(np.mean(f1_pam50_rf), 4)} \u00B1  {round(np.std(f1_pam50_rf), 4)}')
print(f'Mean MCC: {round(np.mean(mcc_pam50_rf), 4)} \u00B1  {round(np.std(mcc_pam50_rf), 4)}')

Mean Accuracy: 0.9442 ± 0.0073
Mean Precision: 0.9544 ±  0.0141
Mean Recall: 0.9289 ±  0.0128
Mean F1: 0.9404 ±  0.0133
Mean MCC: 0.9119 ±  0.0115


#### 2.2 SVM

In [9]:
random.seed(seed)
np.random.seed(seed)

fold = 1
f1_pam50_svm = []
acc_pam50_svm = []
prec_pam50_svm = []
recall_pam50_svm = []
mcc_pam50_svm = []

for train_index, val_index in stratified_k_fold.split(Y_train, Y_train):
    print(f"Starting fold {fold}\n")
    
    # split the training set into train and validation sets for 5-fold cross validation
    print(f"Splitting training set into train and validation sets...")
    x_train, x_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train, y_val = Y_train.iloc[train_index], Y_train.iloc[val_index]

    print(f"Shape of X_train and X_validation sets before filtering: {x_train.shape} and {x_val.shape}")
    print(f"Shape of y_train and y_validation sets before filtering: {y_train.shape} and {y_val.shape}")
    
     # perform pydeseq2 median of ratios normalization
    logMeans, filteredGenes = deseq2_norm_fit(x_train)
    x_train_deseq2 = deseq2_norm_transform(x_train, logMeans, filteredGenes)[0]
    x_val_deseq2 = deseq2_norm_transform(x_val, logMeans, filteredGenes)[0]

    # perform log2 transformation
    x_train_log2 = np.log2(x_train_deseq2 + 1)
    x_val_log2 = np.log2(x_val_deseq2 + 1)

    # perform standardization
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train_log2)
    x_val_scaled = scaler.transform(x_val_log2)

    x_train_scaled = pd.DataFrame(x_train_scaled, index=x_train_log2.index, columns=x_train_log2.columns)
    x_val_scaled = pd.DataFrame(x_val_scaled, index=x_val_log2.index, columns=x_val_log2.columns)
   
    print(f"Shape of X_train and X_validation sets: {x_train_scaled.shape} and {x_val_scaled.shape}")
    print(f"Shape of y_train and y_validation sets: {y_train.shape} and {y_val.shape}")
    print(f"Columns of x train and x val match: {x_train_scaled.columns.equals(x_val_scaled.columns)}")

    # build svm classifier
    svm = SVC(kernel='rbf', gamma='scale', C=5, random_state=seed)

    # fit the svm model on the training fold sets
    svm.fit(x_train_scaled, y_train.values.ravel())

    # make predictions on the validation fold set
    y_pred = svm.predict(x_val_scaled)

    # calculate metric scores
    mcc = metrics.matthews_corrcoef(y_val.values.ravel(), y_pred)
    f1 = metrics.f1_score(y_val.values.ravel(), y_pred, average='macro')
    recall = metrics.recall_score(y_val.values.ravel(), y_pred, average='macro')
    precision = metrics.precision_score(y_val.values.ravel(), y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_val.values.ravel(), y_pred)
    
    print(metrics.classification_report(y_val.values.ravel(), y_pred, digits=4))

    mcc_pam50_svm.append(mcc)
    f1_pam50_svm.append(f1)
    recall_pam50_svm.append(recall)
    prec_pam50_svm.append(precision)
    acc_pam50_svm.append(accuracy)

    fold += 1

Starting fold 1

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets before filtering: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets before filtering: (1763, 1) and (441, 1)
Shape of X_train and X_validation sets: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
Columns of x train and x val match: True
              precision    recall  f1-score   support

           0     1.0000    0.9348    0.9663        46
           1     0.9474    0.9000    0.9231        40
           2     0.9399    0.9777    0.9584       224
           3     0.9213    0.8931    0.9070       131

    accuracy                         0.9410       441
   macro avg     0.9521    0.9264    0.9387       441
weighted avg     0.9413    0.9410    0.9408       441

Starting fold 2

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets before filtering: (1763, 50) and (441, 50)
Shap

In [10]:
print(f'Mean Accuracy: {round(np.mean(acc_pam50_svm), 4)} \u00B1 {round(np.std(acc_pam50_svm), 4)}')
print(f'Mean Precision: {round(np.mean(prec_pam50_svm), 4)} \u00B1  {round(np.std(prec_pam50_svm), 4)}')
print(f'Mean Recall: {round(np.mean(recall_pam50_svm), 4)} \u00B1  {round(np.std(recall_pam50_svm), 4)}')
print(f'Mean F1: {round(np.mean(f1_pam50_svm), 4)} \u00B1  {round(np.std(f1_pam50_svm), 4)}')
print(f'Mean MCC: {round(np.mean(mcc_pam50_svm), 4)} \u00B1  {round(np.std(mcc_pam50_svm), 4)}')

Mean Accuracy: 0.9496 ± 0.0116
Mean Precision: 0.9499 ±  0.0184
Mean Recall: 0.9435 ±  0.014
Mean F1: 0.9464 ±  0.0151
Mean MCC: 0.9208 ±  0.0182


#### 2.3 Logistic Regression

In [11]:
random.seed(seed)
np.random.seed(seed)

fold = 1
f1_pam50_logreg = []
acc_pam50_logreg = []
prec_pam50_logreg = []
recall_pam50_logreg = []
mcc_pam50_logreg = []

for train_index, val_index in stratified_k_fold.split(Y_train, Y_train):
    print(f"Starting fold {fold}\n")
    
    # split the training set into train and validation sets for 5-fold cross validation
    print(f"Splitting training set into train and validation sets...")
    x_train, x_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train, y_val = Y_train.iloc[train_index], Y_train.iloc[val_index]

    print(f"Shape of X_train and X_validation sets before filtering: {x_train.shape} and {x_val.shape}")
    print(f"Shape of y_train and y_validation sets before filtering: {y_train.shape} and {y_val.shape}")

     # perform deseq2 median of ratios normalization
    logMeans, filteredGenes = deseq2_norm_fit(x_train)
    x_train_deseq2 = deseq2_norm_transform(x_train, logMeans, filteredGenes)[0]
    x_val_deseq2 = deseq2_norm_transform(x_val, logMeans, filteredGenes)[0]

    # perform log base 2 transformation
    x_train_log2 = np.log2(x_train_deseq2 + 1)
    x_val_log2 = np.log2(x_val_deseq2 + 1)

    # perform Standardization
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train_log2)
    x_val_scaled = scaler.transform(x_val_log2)

    x_train_scaled = pd.DataFrame(x_train_scaled, index=x_train_log2.index, columns=x_train_log2.columns)
    x_val_scaled = pd.DataFrame(x_val_scaled, index=x_val_log2.index, columns=x_val_log2.columns)
   
    print(f"Shape of X_train and X_validation sets: {x_train_scaled.shape} and {x_val_scaled.shape}")
    print(f"Shape of y_train and y_validation sets: {y_train.shape} and {y_val.shape}")
    print(f"Columns of x train and x val match: {x_train_scaled.columns.equals(x_val_scaled.columns)}")

    # build logistic regression classifier
    logreg = LogisticRegression(C=0.1, solver='lbfgs', penalty='l2', max_iter=1000, random_state=seed)

    # fit the logistic regression model on the training fold sets
    logreg.fit(x_train_scaled, y_train.values.ravel())

    # make predictions on the validation fold set
    y_pred = logreg.predict(x_val_scaled)

    # calculate metric scores
    mcc = metrics.matthews_corrcoef(y_val.values.ravel(), y_pred)
    f1 = metrics.f1_score(y_val.values.ravel(), y_pred, average='macro')
    recall = metrics.recall_score(y_val.values.ravel(), y_pred, average='macro')
    precision = metrics.precision_score(y_val.values.ravel(), y_pred, average='macro')
    accuracy = metrics.accuracy_score(y_val.values.ravel(), y_pred)
    
    print(metrics.classification_report(y_val.values.ravel(), y_pred, digits=4))

    mcc_pam50_logreg.append(mcc)
    f1_pam50_logreg.append(f1)
    recall_pam50_logreg.append(recall)
    prec_pam50_logreg.append(precision)
    acc_pam50_logreg.append(accuracy)

    fold += 1

Starting fold 1

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets before filtering: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets before filtering: (1763, 1) and (441, 1)
Shape of X_train and X_validation sets: (1763, 50) and (441, 50)
Shape of y_train and y_validation sets: (1763, 1) and (441, 1)
Columns of x train and x val match: True
              precision    recall  f1-score   support

           0     1.0000    0.9565    0.9778        46
           1     0.9737    0.9250    0.9487        40
           2     0.9394    0.9688    0.9538       224
           3     0.9219    0.9008    0.9112       131

    accuracy                         0.9433       441
   macro avg     0.9587    0.9378    0.9479       441
weighted avg     0.9436    0.9433    0.9432       441

Starting fold 2

Splitting training set into train and validation sets...
Shape of X_train and X_validation sets before filtering: (1763, 50) and (441, 50)
Shap

In [12]:
print(f'Mean Accuracy: {round(np.mean(acc_pam50_logreg), 4)} \u00B1 {round(np.std(acc_pam50_logreg), 4)}')
print(f'Mean Precision: {round(np.mean(prec_pam50_logreg), 4)} \u00B1  {round(np.std(prec_pam50_logreg), 4)}')
print(f'Mean Recall: {round(np.mean(recall_pam50_logreg), 4)} \u00B1  {round(np.std(recall_pam50_logreg), 4)}')
print(f'Mean F1: {round(np.mean(f1_pam50_logreg), 4)} \u00B1  {round(np.std(f1_pam50_logreg), 4)}')
print(f'Mean MCC: {round(np.mean(mcc_pam50_logreg), 4)} \u00B1  {round(np.std(mcc_pam50_logreg), 4)}')

Mean Accuracy: 0.9514 ± 0.0128
Mean Precision: 0.958 ±  0.0189
Mean Recall: 0.9481 ±  0.0131
Mean F1: 0.9527 ±  0.0158
Mean MCC: 0.9235 ±  0.0199
